## Notebook 05b -- Rung 4 recalibration
Rung 4 = the rung-3 model + a per-participant linear output correction `true = a + b * pred`, fitted by OLS on one enrolment session of the new participant. Plan: `docs/logs/20261008_124119_rung4_recalibration_plan.md`.
This notebook only produces numbers (`fold_metrics_rung4.json`). Statistics are Phase 3, in `significance_analysis.py`.

Load the shared data (same `FEATURES`, `FEATURE_COLS`, `SUBJECT_FOLDS` as 03/04/05).

In [ ]:
from _common import *

Imports and device. Same stack as `inference_timing.py`.

In [ ]:
import gc
import time
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from xgboost import XGBRegressor
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

Switches. `SMOKE = True` runs Ridge only, on participants 1 and 10 (the two with a missing session), and writes to a separate file so the real output is never touched.

In [ ]:
SMOKE = False

OUT_PATH = CURRENT_FOLDER / ('fold_metrics_rung4_smoke.json' if SMOKE else 'fold_metrics_rung4.json')
SMOKE_PARTICIPANTS = [1, 10]
print('SMOKE:', SMOKE, '->', OUT_PATH.name)

Model builders, copied from `inference_timing.py` unchanged (which copied 03 / 04 / 05). Ridge and XGBoost first.

In [ ]:
def make_ridge(params):
    return Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=params['alpha'])),
    ])


def make_xgboost(params):
    return XGBRegressor(
        **params,
        random_state=RANDOM_SEED,
        device='cuda',
        tree_method='hist',
        n_jobs=-1,
    )

The MLP builder: a small PyTorch network wrapped in a fit / predict class, trained on GPU.

In [ ]:
class MLPTorch(nn.Module):
    def __init__(self, n_features, hidden_dims, dropout):
        super().__init__()
        layers = []
        prev = n_features
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
            prev = h
        layers += [nn.Linear(prev, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


class MLPRegressor:
    def __init__(self, hidden_dims, dropout, lr, weight_decay, batch_size, n_epochs, random_state=RANDOM_SEED):
        self.hidden_dims = hidden_dims
        self.dropout = dropout
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.n_epochs = n_epochs
        self.random_state = random_state

    def fit(self, X, y):
        torch.manual_seed(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        self.model_ = MLPTorch(X.shape[1], self.hidden_dims, self.dropout).to(DEVICE)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.MSELoss()

        X_t = torch.from_numpy(X).to(DEVICE)
        y_t = torch.from_numpy(y).to(DEVICE)
        n = len(X_t)

        self.model_.train()
        for epoch in range(self.n_epochs):
            perm = torch.randperm(n, device=DEVICE)
            for start in range(0, n, self.batch_size):
                idx = perm[start:start + self.batch_size]
                opt.zero_grad()
                pred = self.model_(X_t[idx])
                loss = loss_fn(pred, y_t[idx])
                loss.backward()
                opt.step()
        del X_t, y_t
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=np.float32)
        device = next(self.model_.parameters()).device
        X_t = torch.from_numpy(X).to(device)
        self.model_.eval()
        with torch.no_grad():
            pred = self.model_(X_t)
        return pred.cpu().numpy()


def hidden_dims_from(width, n_layers):
    dims = []
    w = width
    for _ in range(n_layers):
        dims.append(max(w, 8))
        w = max(w // 2, 8)
    return dims


def make_mlp(params):
    hidden_dims = hidden_dims_from(params['width'], params['n_layers'])
    return Pipeline([
        ('scale', StandardScaler()),
        ('mlp', MLPRegressor(
            hidden_dims=hidden_dims,
            dropout=params['dropout'],
            lr=params['lr'],
            weight_decay=params['weight_decay'],
            batch_size=params['batch_size'],
            n_epochs=params['n_epochs'],
            random_state=RANDOM_SEED,
        )),
    ])

The three models and their rung-3 result files. In smoke mode only Ridge stays.

In [ ]:
MODELS = {
    'Ridge': ('fold_metrics_ridge_rung3.json', make_ridge),
    'XGBoost': ('fold_metrics_xgboost_rung3.json', make_xgboost),
    'MLP': ('fold_metrics_mlp_rung3.json', make_mlp),
}
if SMOKE:
    MODELS = {'Ridge': MODELS['Ridge']}

print(list(MODELS))

Session 1 is the primary enrolment session, so it must exist for every participant. Participant 001 has no session 4 and participant 010 has no session 6, so the session lists come from the manifest, never from `range(1, 11)`.

In [ ]:
SESSIONS = {int(p.removeprefix('participant')): sorted(v['sessions']) for p, v in MANIFEST['participants'].items()}
for p, sess in SESSIONS.items():
    assert 1 in sess, p
print(SESSIONS)

Resume support. If the output file exists, load it and skip every (model, participant) pair already in it. The file is rewritten after each pair.

In [ ]:
if OUT_PATH.exists():
    with open(OUT_PATH) as f:
        records = json.load(f)
else:
    records = []

done = {(r['model'], r['participant_id']) for r in records}
print(len(records), 'records already on disk;', len(done), 'pairs done')

Main loop. For each model and held-out participant:
1. Refit the rung-3 model on the other 9 participants with the stored `best_params`, and check it reproduces the stored RMSE to 4 decimals.
2. Predict every session of the held-out participant once.
3. Check the pooled uncorrected RMSE over sessions equals the stored RMSE (the bookkeeping identity).
4. For each enrolment session `k`, fit `true = a + b*pred` by plain OLS on all frames of `k`, then score corrected and uncorrected predictions on every other session.

`b <= 0` is flagged as a failed enrolment and is never repaired.

In [ ]:
for model_name, (json_name, make_model) in MODELS.items():
    with open(CURRENT_FOLDER / json_name) as f:
        fold_records = {r['participant_id']: r for r in json.load(f)}

    for fold in SUBJECT_FOLDS:
        test_p = int(fold['test_participant'].removeprefix('participant'))
        if SMOKE and test_p not in SMOKE_PARTICIPANTS:
            continue
        if (model_name, test_p) in done:
            continue
        record = fold_records[test_p]
        params = record['best_params']
        json_rmse = record['metrics']['overall']['rmse']

        # 1. refit on the same training rows as inference_timing.py / the notebooks
        train_nums = [int(p.removeprefix('participant')) for p in fold['train_participants']]
        train_mask = FEATURES['participant_id'].isin(train_nums)
        test_mask = FEATURES['participant_id'] == test_p
        assert not (FEATURES.loc[train_mask, 'participant_id'] == test_p).any()   # leakage: no held-out rows in training

        X_train = FEATURES.loc[train_mask, FEATURE_COLS].to_numpy()
        y_train = FEATURES.loc[train_mask, 'pct_mvc_corr'].to_numpy()
        X_test = FEATURES.loc[test_mask, FEATURE_COLS].to_numpy()
        y_test = FEATURES.loc[test_mask, 'pct_mvc_corr'].to_numpy()
        sess_test = FEATURES.loc[test_mask, 'session_id'].to_numpy()

        t0 = time.time()
        model = make_model(params)
        model.fit(X_train, y_train)
        fit_s = time.time() - t0
        del X_train, y_train
        gc.collect()

        # 2. refit check: same model as rung 3?
        pred_test = np.asarray(model.predict(X_test), dtype=float)
        refit_rmse = float(np.sqrt(np.mean((pred_test - y_test) ** 2)))
        if round(refit_rmse, 4) != round(json_rmse, 4):
            raise RuntimeError(f'{model_name} p{test_p}: refit rmse {refit_rmse:.6f} != json rmse {json_rmse:.6f}')

        # 3. per-session predictions of the held-out participant
        assert sorted(np.unique(sess_test).tolist()) == SESSIONS[test_p]
        pred_s = {}
        true_s = {}
        for s in SESSIONS[test_p]:
            m = sess_test == s
            pred_s[s] = pred_test[m]
            true_s[s] = y_test[m]

        # 4. identity check: per-session uncorrected rmse pools back to the stored rung-3 rmse
        n_list = [len(true_s[s]) for s in SESSIONS[test_p]]
        rmse_list = [float(np.sqrt(np.mean((pred_s[s] - true_s[s]) ** 2))) for s in SESSIONS[test_p]]
        assert sum(n_list) == record['metrics']['overall']['n'] == len(y_test)
        pooled = float(np.sqrt(sum(n * r ** 2 for n, r in zip(n_list, rmse_list)) / sum(n_list)))
        assert round(pooled, 4) == round(json_rmse, 4), (model_name, test_p, pooled, json_rmse)

        # 5. rotation over every enrolment session k
        new_records = []
        for k in SESSIONS[test_p]:
            b, a = np.polyfit(pred_s[k], true_s[k], 1)       # plain OLS, no robust fit, no constraint
            a = float(a)
            b = float(b)

            test_list = [s for s in SESSIONS[test_p] if s != k]
            assert k not in test_list                         # leakage: enrolment session never scored
            test_entries = []
            for j in test_list:
                corrected = a + b * pred_s[j]
                err4 = corrected - true_s[j]
                err3 = pred_s[j] - true_s[j]
                test_entries.append({
                    'session': int(j),
                    'n': int(len(true_s[j])),
                    'rmse_rung4': float(np.sqrt(np.mean(err4 ** 2))),
                    'mae_rung4': float(np.mean(np.abs(err4))),
                    'rmse_rung3': float(np.sqrt(np.mean(err3 ** 2))),
                    'mae_rung3': float(np.mean(np.abs(err3))),
                })

            new_records.append({
                'model': model_name,
                'participant_id': int(test_p),
                'enrol_session': int(k),
                'a': a,
                'b': b,
                'failed_enrolment': bool(b <= 0),
                'refit_rmse': refit_rmse,
                'json_rmse': float(json_rmse),
                'test': test_entries,
            })

        records.extend(new_records)
        with open(OUT_PATH, 'w') as f:
            json.dump(records, f, indent=2)
        done.add((model_name, test_p))

        n_failed = sum(r['failed_enrolment'] for r in new_records)
        print(f"{model_name:8s} p{test_p:02d}  refit={refit_rmse:.4f} json={json_rmse:.4f} pooled={pooled:.4f}  "
              f"turns={len(new_records)}  failed={n_failed}  (fit {fit_s:.0f}s)")

        del model, X_test, y_test, pred_test
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"done -- {len(records)} records in {OUT_PATH.name}")

Plot `a` (shift) and `b` (stretch) against the enrolment session: one line per participant, one panel per model. A flat line means the calibration stayed stable across the visit. It reads the JSON, so it also works after a restart.

In [ ]:
with open(OUT_PATH) as f:
    res = pd.DataFrame(json.load(f))

models_present = [m for m in MODELS if m in set(res['model'])]
fig, axes = plt.subplots(2, len(models_present), figsize=(5 * len(models_present), 7), sharex=True, squeeze=False)
for col, model_name in enumerate(models_present):
    sub = res[res['model'] == model_name]
    for p, g in sub.groupby('participant_id'):
        g = g.sort_values('enrol_session')
        axes[0, col].plot(g['enrol_session'], g['a'], marker='o', label=f'p{p:03d}')
        axes[1, col].plot(g['enrol_session'], g['b'], marker='o', label=f'p{p:03d}')
    axes[1, col].axhline(0, color='k', lw=0.8)
    axes[1, col].axhline(1, color='gray', lw=0.8, ls='--')
    axes[0, col].set_title(model_name)
    axes[0, col].set_ylabel('a (shift, %MVC)')
    axes[1, col].set_ylabel('b (stretch)')
    axes[1, col].set_xlabel('enrolment session')
axes[0, 0].legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.show()